# 📌 Topic 6: Recurrent Neural Networks (RNNs) & LSTMs
> **Deep Learning Crash Course — Part 6**

In this notebook, we will cover **Sequential Data & Time-Series**:
1. Why sequential data requires memory (text, audio, time-series).
2. Standard RNNs & Vanishing Gradient Problems.
3. **LSTM (Long Short-Term Memory)** Gating Mechanisms:
   - **Forget Gate** ($f_t$), **Input Gate** ($i_t$), **Output Gate** ($o_t$).
4. PyTorch LSTM for Sine Wave forecasting.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


### 6.1 PyTorch LSTM Model


In [ ]:
class LSTMForecaster(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=1):
        super(LSTMForecaster, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
        
    def forward(self, x):
        # x: [Batch, Seq_Len, Features]
        lstm_out, _ = self.lstm(x)
        last_step = lstm_out[:, -1, :]
        return self.fc(last_step)

# Generate Sine Wave Sequence
t = np.linspace(0, 50, 400)
sine = np.sin(t)

seq_len = 20
X_seq, y_seq = [], []
for i in range(len(sine) - seq_len):
    X_seq.append(sine[i:i+seq_len])
    y_seq.append(sine[i+seq_len])

X_seq = np.array(X_seq)[..., np.newaxis]
y_seq = np.array(y_seq)[..., np.newaxis]

X_t = torch.FloatTensor(X_seq).to(device)
y_t = torch.FloatTensor(y_seq).to(device)

model = LSTMForecaster().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.MSELoss()

# Train
for epoch in range(120):
    model.train()
    optimizer.zero_grad()
    preds = model(X_t)
    loss = criterion(preds, y_t)
    loss.backward()
    optimizer.step()

model.eval()
with torch.no_grad():
    predictions = model(X_t).cpu().numpy()

plt.figure(figsize=(12, 4))
plt.plot(y_seq, label="Actual Sine Values", color="black", lw=2)
plt.plot(predictions, label="LSTM Forecast", color="red", linestyle="--", lw=2)
plt.title("LSTM Sine Wave Time-Series Forecasting")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
